# Cell 1: Imports and verify files

In [ ]:
# Day 10: Apo D2 Receptor Production MD
# Long simulation of D2 in DPPC membrane WITHOUT any drug
# This is your baseline for all drug-binding comparisons

import os
import time
import numpy as np
import matplotlib.pyplot as plt
from sys import stdout

from openmm import *
from openmm.app import *
from openmm.unit import *

os.chdir('/workspace/d2_system')
print(f"Working in: {os.getcwd()}")

# Verify the equilibrated files exist
required = ['day9_final_equilibrated.xml', 'day9_equilibrated_topology.pdb']
for f in required:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"  ✓ {f} ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ MISSING: {f}")
        
print(f"\nPlatforms: {[Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]}")
print("Day 10 imports OK")

# Cell 2: Load the equilibrated system

In [ ]:
# === LOAD DAY 9 EQUILIBRATED SYSTEM ===

print("Loading topology PDB...")
topology_pdb = PDBFile('day9_equilibrated_topology.pdb')
print(f"  Topology: {topology_pdb.topology.getNumAtoms()} atoms")

# Important: we need to rebuild the system XML because the equilibrated state
# was saved with restraints still in the system. Let's rebuild from the original
# converted system to ensure no leftover restraints.

print("\nLoading OpenMM system XML...")
with open('d2_membrane_system.xml', 'r') as f:
    system = XmlSerializer.deserialize(f.read())
print(f"  System: {system.getNumParticles()} particles, {system.getNumForces()} forces")

# Add membrane barostat (the system was saved before barostat was added in Day 9)
print("\nAdding MonteCarloMembraneBarostat (XYIsotropic, ZFree, 310 K, 1 atm)...")
barostat = MonteCarloMembraneBarostat(
    1.0*bar,
    0.0*bar*nanometer,
    310*kelvin,
    MonteCarloMembraneBarostat.XYIsotropic,
    MonteCarloMembraneBarostat.ZFree,
)
system.addForce(barostat)
print(f"  Total forces now: {system.getNumForces()}")

# Cell 3: Set up the production simulation and load the equilibrated state

In [ ]:
# === SET UP PRODUCTION SIMULATION ===

# Production integrator
prod_integrator = LangevinMiddleIntegrator(310*kelvin, 1/picosecond, 0.002*picoseconds)
platform = Platform.getPlatformByName('CUDA')
properties = {'Precision': 'mixed'}

# Build simulation
sim = Simulation(topology_pdb.topology, system, prod_integrator, platform, properties)

# Load the equilibrated state from Day 9
# This restores positions, velocities, and box vectors from the equilibrated system
print("Loading equilibrated state from Day 9...")
try:
    sim.loadState('day9_final_equilibrated.xml')
    print("  Loaded successfully")
except Exception as e:
    print(f"  ⚠ Error loading state: {e}")
    print("  Falling back: setting positions from topology PDB")
    sim.context.setPositions(topology_pdb.positions)
    sim.context.setPeriodicBoxVectors(*topology_pdb.topology.getPeriodicBoxVectors())
    sim.context.setVelocitiesToTemperature(310*kelvin)

# Verify the state is good
state = sim.context.getState(getEnergy=True)
E = state.getPotentialEnergy().value_in_unit(kilojoule_per_mole)
print(f"\nInitial potential energy: {E:,.1f} kJ/mol")
print(f"(Should be around -1,034,000 kJ/mol if state loaded correctly)")

# Check box dimensions
box = state.getPeriodicBoxVectors()
print(f"\nBox dimensions:")
print(f"  X: {box[0][0].value_in_unit(nanometer):.3f} nm")
print(f"  Y: {box[1][1].value_in_unit(nanometer):.3f} nm")
print(f"  Z: {box[2][2].value_in_unit(nanometer):.3f} nm")

# Cell 4: Set up reporters and launch 50 ns production

In [ ]:
# === SET UP REPORTERS FOR PRODUCTION ===

# Trajectory: save every 50 ps (every 25,000 steps × 2 fs)
# For 50 ns total: 50,000 / 50 = 1000 frames
# DCD file size estimate: 1000 frames × 118,026 atoms × 3 floats × 4 bytes = ~1.4 GB
print("Setting up trajectory reporter (every 50 ps, ~1000 frames over 50 ns)...")
sim.reporters.append(DCDReporter('day10_apo_d2_50ns.dcd', reportInterval=25000))

# Metrics CSV every 50 ps
sim.reporters.append(StateDataReporter(
    'day10_apo_d2_50ns_metrics.csv',
    reportInterval=25000,
    step=True, time=True,
    potentialEnergy=True, kineticEnergy=True, totalEnergy=True,
    temperature=True, volume=True, density=True
))

# Live progress every 500 ps (every 250,000 steps)
# 50 ns / 500 ps = 100 progress lines total
total_steps = 25_000_000  # 50 ns × 500 ps/ns × 1000 steps/ps / 2 fs
sim.reporters.append(StateDataReporter(
    stdout, reportInterval=250000,
    step=True, time=True,
    temperature=True, density=True, speed=True,
    progress=True, remainingTime=True,
    totalSteps=total_steps + sim.currentStep,
    separator='\t'
))

# Save state every 5 ns (checkpoint, in case we need to resume)
# 5 ns = 2,500,000 steps
sim.reporters.append(CheckpointReporter(
    'day10_checkpoint.chk',
    reportInterval=2500000
))

print(f"Reporters set:")
print(f"  Trajectory: day10_apo_d2_50ns.dcd (every 50 ps)")
print(f"  Metrics:    day10_apo_d2_50ns_metrics.csv (every 50 ps)")
print(f"  Checkpoint: day10_checkpoint.chk (every 5 ns)")

print(f"\n{'='*60}")
print(f"STARTING 50 ns APO D2 PRODUCTION MD")
print(f"Estimated wall time: 7.5-8 hours at 155 ns/day")
print(f"Progress updates every 500 ps ({total_steps//250000} updates total)")
print(f"{'='*60}\n")

t_start = time.time()
sim.step(total_steps)
elapsed = time.time() - t_start

print(f"\n{'='*60}")
print(f"PRODUCTION COMPLETE")
print(f"Wall time: {elapsed:.1f} seconds = {elapsed/3600:.2f} hours")
print(f"Performance: {(50/elapsed)*86400:.1f} ns/day")
print(f"{'='*60}")

# Final state save
sim.saveState('day10_apo_d2_50ns_final.xml')
print("\nFinal state saved: day10_apo_d2_50ns_final.xml")